In [ ]:
# @title 📂 AI Hub 데이터 4대 권역별 자동 분류기
import os
import json
import shutil
from tqdm import tqdm

# ==========================================
# 🔧 1. 경로 설정 (본인의 드라이브 경로로 수정 필수!)
# ==========================================
# 압축 풀린 원본 데이터가 있는 위치
BASE_DIR = "/content/drive/MyDrive/HabitLink/AIHub_Val"
SOURCE_DIR = os.path.join(BASE_DIR, "Source") # wav 파일들이 있는 곳
LABEL_DIR = os.path.join(BASE_DIR, "Label")   # json 파일들이 있는 곳

# 분류된 파일이 저장될 최종 목적지
DEST_DIR = "/content/drive/MyDrive/HabitLink/dialect_dataset_final"

In [ ]:
# ==========================================
# 🔧 2. 지역 코드 매핑 (AI Hub 표준 코드 기준)
# 설명서의 지역 구분에 따라 그룹핑합니다.
# ==========================================
REGION_MAP = {
    # [표준어권] 서울/인천/경기
    "01": "standard",

    # [경상권] 부산/대구/울산/경상
    "05": "gyeongsang",

    # [전라/제주권] 광주/전라/제주
    "04": "jeolla_jeju",
    "06": "jeolla_jeju", # 제주는 보통 06번인 경우가 많음 (확인 후 자동 통합)

    # [충청/강원권] 대전/세종/충청/강원
    "02": "chungcheong_gangwon", # 강원
    "03": "chungcheong_gangwon"  # 충청
}

def organize_dataset():
    # 1. 목적지 폴더 생성
    folders = ["standard", "gyeongsang", "jeolla_jeju", "chungcheong_gangwon"]
    for folder in folders:
        os.makedirs(os.path.join(DEST_DIR, folder), exist_ok=True)

    print("🚀 데이터 분류 준비 중...")

    # 2. WAV 파일 위치 미리 매핑 (속도 최적화)
    # (폴더 깊이가 복잡할 수 있으므로 미리 위치를 찾아둠)
    wav_path_map = {}
    print("   Scanning WAV files...", end=" ")
    for root, dirs, files in os.walk(SOURCE_DIR):
        for f in files:
            if f.lower().endswith(".wav"):
                wav_path_map[f] = os.path.join(root, f)
    print(f"Done! ({len(wav_path_map)} files found)")

    # 3. JSON 라벨 읽기 및 분류 시작
    print("\n🚀 분류 및 복사 시작!")

    stats = {k: 0 for k in folders}
    stats["unknown"] = 0
    stats["missing_wav"] = 0

    # JSON 폴더 탐색
    json_files = []
    for root, dirs, files in os.walk(LABEL_DIR):
        for f in files:
            if f.lower().endswith(".json"):
                json_files.append(os.path.join(root, f))

    for json_file in tqdm(json_files):
        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                data = json.load(f)

            # JSON 파싱 (설명서 구조 기반)
            # Speaker -> Region 정보 확인
            region_code = data.get("Speaker", {}).get("Region")
            file_name = data.get("File", {}).get("FileName")

            if not region_code or not file_name:
                continue

            # 타겟 폴더 결정
            target_folder = REGION_MAP.get(region_code)

            if target_folder:
                # WAV 파일 찾기
                src_wav = wav_path_map.get(file_name)

                if src_wav:
                    dst_wav = os.path.join(DEST_DIR, target_folder, file_name)

                    # 파일 복사 (이미 있으면 건너뜀)
                    if not os.path.exists(dst_wav):
                        shutil.copy2(src_wav, dst_wav)

                    stats[target_folder] += 1
                else:
                    stats["missing_wav"] += 1
            else:
                # 매핑되지 않은 지역 코드 발견 시 (디버깅용)
                # print(f"Unknown Region Code: {region_code}")
                stats["unknown"] += 1

        except Exception as e:
            print(f"Error: {e}")
            continue

    print("\n✅ 분류 완료! 통계:")
    for key, value in stats.items():
        print(f"  - {key}: {value}개")
    print(f"\n📂 저장 위치: {DEST_DIR}")

# 실행
organize_dataset()